# 🔧 Feature Engineering — Manufacturing Predictive Maintenance

Aggregates Silver sensor readings per machine per day and joins with
failure events to create features for maintenance-risk prediction.

**Reads from:** `silver_sensor_readings`, `silver_production_batches`, `silver_equipment_catalog`

**Writes to:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, stddev, count,
    sum as spark_sum, max as spark_max, min as spark_min,
    to_date, datediff, abs as spark_abs, dayofweek, month
)

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load Silver tables (created by batch/02_silver_transform)
sensor_df = spark.read.table('silver_sensor_readings')
batch_df = spark.read.table('silver_production_batches')
equip_df = spark.read.table('silver_equipment_catalog')

print(f'Silver sensor readings: {sensor_df.count():,} rows')
print(f'Silver production batches: {batch_df.count():,} rows')
print(f'Silver equipment catalog: {equip_df.count():,} rows')

required_batch_cols = {'machine_id', 'batch_start', 'failure_event'}
missing = required_batch_cols - set(batch_df.columns)
if missing:
    raise ValueError(
        f"silver_production_batches missing columns {sorted(missing)}. "
        "Regenerate manufacturing-qc/data/*.csv using _generate_data.py and rerun bronze/silver notebooks."
    )

In [ ]:
# === Sensor features: aggregate per machine per day ===
sensor_daily = (
    sensor_df
    .withColumn('sensor_date', to_date('reading_date'))
    .groupBy('machine_id', 'production_line', 'sensor_date')
    .agg(
        avg('temperature').alias('avg_temp'),
        stddev('temperature').alias('std_temp'),
        spark_max('temperature').alias('max_temp'),
        spark_min('temperature').alias('min_temp'),
        avg('pressure').alias('avg_pressure'),
        stddev('pressure').alias('std_pressure'),
        spark_max('pressure').alias('max_pressure'),
        spark_min('pressure').alias('min_pressure'),
        avg('vibration').alias('avg_vibration'),
        stddev('vibration').alias('std_vibration'),
        spark_max('vibration').alias('max_vibration'),
        spark_min('vibration').alias('min_vibration'),
        avg('humidity').alias('avg_humidity'),
        count('*').alias('reading_count'),
        spark_sum(when(col('temp_anomaly'), 1).otherwise(0)).alias('temp_anomaly_count'),
        spark_sum(when(col('vibration_anomaly'), 1).otherwise(0)).alias('vibration_anomaly_count'),
    )
    .withColumn('temp_range', col('max_temp') - col('min_temp'))
    .withColumn('pressure_range', col('max_pressure') - col('min_pressure'))
    .withColumn('vibration_range', col('max_vibration') - col('min_vibration'))
    .withColumn('anomaly_ratio', (col('temp_anomaly_count') + col('vibration_anomaly_count')) / col('reading_count'))
)

print(f'Sensor daily features: {sensor_daily.count():,} rows')

In [ ]:
# === Failure target: did this machine have a failure_event on this day? ===
# Be explicit about type: CSV inference can load failure_event as string in some runs.
batch_with_flag = batch_df.withColumn('failure_event_int', col('failure_event').cast('int'))

failure_events = (
    batch_with_flag
    .withColumn('batch_date', to_date('batch_start'))
    .filter(col('failure_event_int') == 1)
    .groupBy('machine_id', 'batch_date')
    .agg(
        count('*').alias('failure_event_count'),
        spark_sum('downtime_minutes').alias('total_failure_downtime'),
        spark_sum('defect_count').alias('total_failure_defects'),
    )
    .withColumn('had_failure', lit(1))
)

failure_days = failure_events.count()
print(f'Days with failure events: {failure_days:,}')

In [ ]:
# === Join sensor features with failure target + equipment attributes ===
ml_features = (
    sensor_daily
    .join(
        failure_events.select(
            'machine_id',
            col('batch_date').alias('sensor_date'),
            'had_failure',
            'failure_event_count',
            'total_failure_downtime',
            'total_failure_defects',
        ),
        ['machine_id', 'sensor_date'],
        'left'
    )
    .join(
        equip_df.select('machine_id', 'machine_type', 'install_date'),
        'machine_id',
        'left'
    )
    .withColumn('had_failure', when(col('had_failure').isNull(), 0).otherwise(1))
    .withColumn('equipment_age_days', datediff(col('sensor_date'), to_date('install_date')))
    .withColumn('day_of_week', dayofweek('sensor_date'))
    .withColumn('month', month('sensor_date'))
    .drop('install_date')
    .na.fill(0)
    .na.fill('unknown', subset=['machine_type'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_failure') == 1).count()
failure_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

# Guardrail: if positives collapse unexpectedly, fail fast instead of silently training garbage.
if total_rows > 0 and positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} positive rows '
        f'({failure_rate:.2f}%). Check failure_event typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')

print(f'\nGold ML features written: {total_rows:,} rows')
print(f'Columns: {len(ml_features.columns)}')
print(f'Failure rate: {failure_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')